In [24]:
import sys
from pathlib import Path

# Adiciona o diretório raiz (projeto_tcc) ao PATH
project_dir = Path.cwd().parent  # assume que o notebook está em projeto_tcc/notebooks/
sys.path.append(str(project_dir))

import polars as pl
from scripts.configs import DIRS
from scripts.utils import ler_arquivo_polars

ModuleNotFoundError: No module named 'scripts.configs'

In [ ]:

# Carregar tabelas para análise
cidades = ler_arquivo_polars(f"{DIRS['FINAL_CIDADES']}/cidades_ibge_2022.parquet")
mortalidade = ler_arquivo_polars(f"{DIRS['FINAL_MORTALIDADE']}/mortalidade_2022.parquet")
estabelecimentos = ler_arquivo_polars(f"{DIRS['CONCAT_CNES']}/tbestabelecimento_2022.parquet")

# ------------------------
# Análise Cidades IBGE
# ------------------------
print("\nResumo cidades IBGE")
print(cidades.describe())

print("\nUFs com maior e menor IDH")
print(cidades.select(["uf", "nome", "idhm"]).sort("idhm", descending=True).head(5))
print(cidades.select(["uf", "nome", "idhm"]).sort("idhm").head(5))

# ------------------------
# Análise Mortalidade
# ------------------------
print("\nResumo mortalidade por município")
mortalidade_por_mun = (
    mortalidade.groupby(["CODMUNOCOR", "NO_MUNICIPIO", "CO_SIGLA_ESTADO"])
    .agg(pl.count().alias("qtd_obitos"))
    .sort("qtd_obitos", descending=True)
)
print(mortalidade_por_mun.head(5))

# ------------------------
# Análise Estabelecimentos
# ------------------------
print("\nQuantidade de estabelecimentos por município")
estabs_por_mun = (
    estabelecimentos.groupby(["CO_MUNICIPIO_GESTOR"])
    .agg(pl.count().alias("qtd_estabs"))
    .sort("qtd_estabs", descending=True)
)
print(estabs_por_mun.head(5))

# ------------------------
# Exemplo de Join cruzado (obitos por estab)
# ------------------------
obitos_por_estab = (
    mortalidade.filter(pl.col("CODESTAB").is_not_null())
    .groupby("CODESTAB")
    .agg(pl.count().alias("qtd_obitos_estab"))
    .sort("qtd_obitos_estab", descending=True)
)
print("\nEstabelecimentos com mais óbitos registrados")
print(obitos_por_estab.head(5))